In [1]:
# Model version: Tue May 5 19:35:22 CEST 2026
# =========================
# TRAINING PHASE
# =========================
# This cell defines and trains the model.
#
# - The model is trained exclusively on the training dataset.
# - Model checkpoints are saved every 5 epochs.
#
# Training data:
# - Inputs:  stochastic_tf_input_XXXX.npy
# - Outputs: stochastic_tf_output_XXXX.npy
# These files are part of the "stochastic_model" dataset.
#
# Validation and test data:
# - stochastic_input.npy
# - stochastic_output.npy
# These contain 10,000 samples independent from the training set.
# They are later split into:
#   - 5,000 samples for validation
#   - 5,000 samples for testing
#
# Important:
# - Validation and test datasets are NOT used during training.
# - All evaluation and benchmarking are performed in later sections.

import tensorflow as tf
import numpy as np

class DimensionScaling(tf.keras.layers.Layer):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def build(self, input_shape):
        self.alpha = self.add_weight(shape=(self.dim,),initializer="ones",trainable=True,name="alpha")
    def call(self, inputs):
        return inputs*self.alpha
class SaveEveryNEpochs(tf.keras.callbacks.Callback):
    def __init__(self, n):
        self.n = n

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.n == 0:
            self.model.save(f"model_epoch_{epoch+1}.keras")

def dataset_from_file(i):
    X = np.load(f"/kaggle/input/datasets/jdoesagazl/stochastic-model/stochastic_tf_input_{i:04d}.npy").astype(np.float32)
    Y = np.load(f"/kaggle/input/datasets/jdoesagazl/stochastic-model/stochastic_tf_output_{i:04d}.npy")
    Y = ((Y[:, None] >> np.arange(48)) & 1).astype(np.float32)
    return tf.data.Dataset.from_tensor_slices((X, Y))

print(tf.config.list_physical_devices('GPU'))

inputs = tf.keras.Input(shape=(390,)) # redundant in theory but not in practice
x = DimensionScaling(390)(inputs) # scale so that context and vectors have the same contribution

# Interaction block
x = tf.keras.layers.Dense(390, activation='relu')(x)
x = tf.keras.layers.Dense(390, activation='relu')(x)
x = tf.keras.layers.Dense(390, activation=None)(x)
x = tf.keras.layers.LayerNormalization()(x)
x = tf.keras.layers.Activation('gelu')(x)

x = tf.keras.layers.Dense(20<<8, activation='relu')(x) # 256 of dimension 20

for N in range(7, 1, -1):
    x = tf.keras.layers.Reshape((1<<N, 2, 20))(x)
    x1 = x[:, :, 0, :]   # (batch, groups, 20)
    x2 = x[:, :, 1, :]   # (batch, groups, 20)
    
    concat = tf.keras.layers.Concatenate()([x1, x2])

    h = tf.keras.layers.Dense(40, activation='relu')(concat)

    gate = tf.keras.layers.Dense(20, activation='sigmoid')(h)
    candidate = tf.keras.layers.Dense(20)(h)

    merged = gate * candidate

    x = tf.keras.layers.Activation('relu')(merged)


# Now we have 4 vectors of dimension 20 which I will reshape as 2 of dimension 40
x = tf.keras.layers.Reshape((2,40))(x)
x = tf.keras.layers.Dense(24, activation='relu')(x) # 2 of dimension 24 instead of 20
x = tf.keras.layers.Reshape((48,))(x)
outputs = tf.keras.layers.Dense(48, activation='sigmoid')(x)# output layer
model = tf.keras.Model(inputs, outputs)

datasets = [dataset_from_file(i) for i in range(11)]
dataset = tf.data.Dataset.sample_from_datasets(datasets)

dataset = dataset.shuffle(20000)
dataset = dataset.batch(512)
dataset = dataset.cache()
dataset = dataset.prefetch(tf.data.AUTOTUNE)
#dataset = dataset.repeat()
checkpoint = SaveEveryNEpochs(5)
model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-3),
    loss='binary_crossentropy',
    metrics=[ tf.keras.metrics.BinaryAccuracy(threshold=0.5)],
)

model.fit(dataset, callbacks=[checkpoint], epochs=30)


2026-05-23 13:52:57.444185: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779544377.638085      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779544377.704567      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779544378.170892      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779544378.170936      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779544378.170939      22 computation_placer.cc:177] computation placer alr

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


I0000 00:00:1779544405.129823      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779544405.135805      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/30


I0000 00:00:1779544472.835954      68 service.cc:152] XLA service 0x792040009960 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779544472.836014      68 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779544472.836022      68 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779544474.257554      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


     13/Unknown 18s 14ms/step - binary_accuracy: 0.5810 - loss: 0.6918

I0000 00:00:1779544479.239124      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


6350/6350 ━━━━━━━━━━━━━━━━━━━━ 136s 19ms/step - binary_accuracy: 0.6594 - loss: 0.6039
Epoch 2/30
  13/6350 ━━━━━━━━━━━━━━━━━━━━ 1:22 13ms/step - binary_accuracy: 0.6796 - loss: 0.5815

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - binary_accuracy: 0.6782 - loss: 0.5825
Epoch 3/30
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - binary_accuracy: 0.6820 - loss: 0.5777
Epoch 4/30
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - binary_accuracy: 0.6844 - loss: 0.5747
Epoch 5/30
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - binary_accuracy: 0.6862 - loss: 0.5723
Epoch 6/30
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - binary_accuracy: 0.6877 - loss: 0.5703
Epoch 7/30
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - binary_accuracy: 0.6889 - loss: 0.5686
Epoch 8/30
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - binary_accuracy: 0.6899 - loss: 0.5670
Epoch 9/30
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - binary_accuracy: 0.6909 - loss: 0.5656
Epoch 10/30
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - binary_accuracy: 0.6917 - loss: 0.5642
Epoch 11/30
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - binary_accuracy: 0.6925 - loss: 0.5630
Epoch 12/30
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 57s 9m

In [2]:
# =========================
# BIT-LEVEL ACCURACY
# =========================
# This cell computes the average accuracy of each individual output bit
# for every saved model checkpoint.
#
# - For each checkpoint, bit accuracy is computed over the entire training dataset.
# - The result is stored as: bit_accuracy_<epoch>.npy
#
# Purpose:
# - These bit-level accuracies are later used to define weighting coefficients:
#     alpha = bit_accuracy - 0.5
# - The weights are applied in the scoring function used to rank candidate
#   codewords during inference.
#
# Important:
# - This is NOT a model evaluation step.
# - No validation or test data is used here.
# - This is a diagnostic/statistical analysis used to parameterize the decoding stage.
path_data = '/kaggle/input/datasets/jdoesagazl/stochastic-model/'
model_path = "/kaggle/working/"
models = list(range(5, 31, 5))  # epochs 5, 10, 15, 20, 25, 30

for m in models:
    print(f"Computing bit accuracy for epoch {m}")
    model.load_weights(f"{model_path}model_epoch_{m}.keras")
    bit_accuracy = np.zeros(48)
    N = 0

    for i in range(13):
        X = np.load(f"{path_data}stochastic_tf_input_{i:04d}.npy").astype(np.float32)
        Y = np.load(f"{path_data}stochastic_tf_output_{i:04d}.npy")
        Y_bits = ((Y[:, None] >> np.arange(48)) & 1).astype(np.float32)
        preds = model.predict(X, verbose=0)
        bit_accuracy += ((preds > 0.5) == Y_bits).mean(axis=0) * X.shape[0]
        N += X.shape[0]

    bit_accuracy /= N
    np.save(f'{model_path}bit_accuracy_{m}.npy', bit_accuracy)

Computing bit accuracy for epoch 5
Computing bit accuracy for epoch 10
Computing bit accuracy for epoch 15
Computing bit accuracy for epoch 20
Computing bit accuracy for epoch 25
Computing bit accuracy for epoch 30


In [3]:
# =========================
# VALIDATION & MODEL SELECTION
# =========================
# This cell evaluates all saved model checkpoints on the validation dataset
# to select the best-performing model.
#
# Procedure:
# - For each checkpoint, the corresponding bit_accuracy is loaded.
# - Bit-level weights are computed as:
#     alpha = bit_accuracy - 0.5
# - These weights are used in the scoring function to rank candidate codewords.
# - Top-k prediction accuracy (top-1, top-10) is computed.
#
# Validation data:
# - Source: stochastic_input.npy / stochastic_output.npy
# - The first 5,000 samples are used as the validation dataset.
# - These samples are independent from the training dataset.
#
# Model selection:
# - The checkpoint with the highest top-1 validation accuracy is selected.
# - The selected model is saved as: best_model.keras
# - The corresponding alpha is saved as: best_alpha.npy
#
# Additional data:
# - vocabulary_encoding.npy contains the 48-bit codewords for all vocabulary tokens (~56k).
# - The scoring function evaluates all codewords to determine the most likely predictions.
#
# Important:
# - This step is used for model selection only.
# - The remaining 5,000 samples are reserved for final test evaluation.
# - No test data is used in this stage.
path_encoding = '/kaggle/input/datasets/jdoesagazl/token-encoding/'
window_size = 1000
best_accuracy = 0  
eps = 1e-6

encoding = np.load(f"{path_encoding}vocabulary_encoding.npy")
encoding_bits = ((encoding[:, None] >> np.arange(48)) & 1).astype(np.int8)
print('_'*100)

for m in models:
    accuracy_top_1 = 0
    accuracy_top_10 = 0
    model.load_weights(f"{model_path}model_epoch_{m}.keras")
    bit_accuracy = np.load(f'{model_path}bit_accuracy_{m}.npy')
    #alpha = np.log(bit_accuracy)-np.log(1-bit_accuracy) # performs worse
    alpha = bit_accuracy - 0.5

    for i in range(0, 5000, window_size): # first 5000 are validation dataset
                                           # last 5000 are test dataset
        X = np.load(f"{path_data}stochastic_input.npy").astype(np.float32)
        Y = np.load(f"{path_data}stochastic_output.npy")
        X = X[i:i+window_size].copy()
        Y = Y[i:i+window_size].copy()
        preds = model.predict(X, verbose=0)
        preds = np.clip(preds, eps, 1 - eps)

        logit = np.log(preds) - np.log(1 - preds)  # (N_preds, bits)
        bias  = np.log(1 - preds)                   # (N_preds, bits)

        # apply bit weights
        logit *= alpha
        bias  *= alpha

        # main computation
        scores = logit @ encoding_bits.T + bias.sum(axis=1, keepdims=True)
        result = np.argsort(-scores, axis=1)[:, :10]

        out = np.array([[encoding[result[i][j]] for j in range(result.shape[1])] for i in range(result.shape[0])])

        accuracy_top_1  += (out[:, 0] == Y).mean() * X.shape[0]
        accuracy_top_10 += (out == Y[:, None]).any(axis=1).mean() * X.shape[0]

    accuracy_top_1  /= 5000
    accuracy_top_10 /= 5000
    print(f"Accuracy of top-1 predictions for model {m}: {100*accuracy_top_1:02.2f}%")
    print(f"Accuracy of top-10 predictions for model {m}: {100*accuracy_top_10:02.2f}%")
    print('_'*100)
    print()

    if accuracy_top_1 > best_accuracy:
        best_accuracy = accuracy_top_1
        with open(f'{model_path}best_model.keras', 'wb+') as f1:
            with open(f'{model_path}model_epoch_{m}.keras', 'rb') as f2:
                f1.write(f2.read())
        np.save(f'{model_path}best_alpha.npy', alpha)

____________________________________________________________________________________________________
Accuracy of top-1 predictions for model 5: 17.14%
Accuracy of top-10 predictions for model 5: 25.44%
____________________________________________________________________________________________________

Accuracy of top-1 predictions for model 10: 17.34%
Accuracy of top-10 predictions for model 10: 25.24%
____________________________________________________________________________________________________

Accuracy of top-1 predictions for model 15: 16.92%
Accuracy of top-10 predictions for model 15: 25.38%
____________________________________________________________________________________________________

Accuracy of top-1 predictions for model 20: 16.94%
Accuracy of top-10 predictions for model 20: 25.24%
____________________________________________________________________________________________________

Accuracy of top-1 predictions for model 25: 16.30%
Accuracy of top-10 predictions

In [4]:
# =========================
# ABLATION: ZERO SUBSTITUTION
# =========================
# In this experiment, the first 170 entries of the input vector
# (corresponding to the context representation) are set to zero.
#
# The modified inputs are then evaluated on the test dataset to compute
# top-k prediction accuracy.
# Note: The first 170 input dimensions correspond to the context vector representation.

accuracy_top_1  = 0
accuracy_top_10 = 0
window_size  = 1000
context_start = 170
eps = 1e-6

model.load_weights(f"{model_path}best_model.keras")
alpha         = np.load(f'{model_path}best_alpha.npy')
encoding      = np.load(f"{path_encoding}vocabulary_encoding.npy")
encoding_bits = ((encoding[:, None] >> np.arange(48)) & 1).astype(np.int8)
N = 0

for i in range(5000, 10000, window_size):
    X = np.load(f"{path_data}stochastic_input.npy").astype(np.float32)
    Y = np.load(f"{path_data}stochastic_output.npy")
    X = X[i:i+window_size].copy()
    Y = Y[i:i+window_size].copy()
    X[:, :context_start] = 0  # ablation: zero out context vectors

    preds = model.predict(X, verbose=0)
    preds = np.clip(preds, eps, 1 - eps)

    logit = np.log(preds) - np.log(1 - preds)  # (N_preds, bits)
    bias  = np.log(1 - preds)                   # (N_preds, bits)

    # apply bit weights
    logit *= alpha
    bias  *= alpha

    # compute scores and top-10 predictions
    scores = logit @ encoding_bits.T + bias.sum(axis=1, keepdims=True)
    result = np.argsort(-scores, axis=1)[:, :10]

    out = np.array([[encoding[result[i][j]] for j in range(result.shape[1])] for i in range(result.shape[0])])

    accuracy_top_1  += (out[:, 0] == Y).mean() * X.shape[0]
    accuracy_top_10 += (out == Y[:, None]).any(axis=1).mean() * X.shape[0]
    N += X.shape[0]

accuracy_top_1  /= N
accuracy_top_10 /= N
print(f"Accuracy of top-1 predictions:  {100*accuracy_top_1:02.2f}%")
print(f"Accuracy of top-10 predictions: {100*accuracy_top_10:02.2f}%")

Accuracy of top-1 predictions:  16.60%
Accuracy of top-10 predictions: 23.90%


In [5]:
# =========================
# ABLATION: SHUFFLE
# =========================
# In this experiment, the first 170 entries of the input vector
# (context representation) are randomly shuffled.
#
# The modified inputs are evaluated on the test dataset to compute
# top-k prediction accuracy.

accuracy_top_1  = 0
accuracy_top_10 = 0
window_size   = 1000
context_start = 170
eps = 1e-6

model.load_weights(f"{model_path}best_model.keras")
alpha         = np.load(f'{model_path}best_alpha.npy')
encoding      = np.load(f"{path_encoding}vocabulary_encoding.npy")
encoding_bits = ((encoding[:, None] >> np.arange(48)) & 1).astype(np.int8)
N = 0

for i in range(5000, 10000, window_size):
    X = np.load(f"{path_data}stochastic_input.npy").astype(np.float32)
    Y = np.load(f"{path_data}stochastic_output.npy")
    X = X[i:i+window_size].copy()
    Y = Y[i:i+window_size].copy()

    # ablation: shuffle context vectors across samples
    X[:, :context_start] = X[np.random.permutation(X.shape[0]), :context_start]

    preds = model.predict(X, verbose=0)
    preds = np.clip(preds, eps, 1 - eps)

    logit = np.log(preds) - np.log(1 - preds)  # (N_preds, bits)
    bias  = np.log(1 - preds)                   # (N_preds, bits)

    # apply bit weights
    logit *= alpha
    bias  *= alpha

    # compute scores and top-10 predictions
    scores = logit @ encoding_bits.T + bias.sum(axis=1, keepdims=True)
    result = np.argsort(-scores, axis=1)[:, :10]

    out = np.array([[encoding[result[i][j]] for j in range(result.shape[1])] for i in range(result.shape[0])])

    accuracy_top_1  += (out[:, 0] == Y).mean() * X.shape[0]
    accuracy_top_10 += (out == Y[:, None]).any(axis=1).mean() * X.shape[0]
    N += X.shape[0]

accuracy_top_1  /= N
accuracy_top_10 /= N
print(f"Accuracy of top-1 predictions:  {100*accuracy_top_1:02.2f}%")
print(f"Accuracy of top-10 predictions: {100*accuracy_top_10:02.2f}%")


Accuracy of top-1 predictions:  15.96%
Accuracy of top-10 predictions: 23.32%


In [6]:
# =========================
# ABLATION: MEAN SUBSTITUTION
# =========================
# In this experiment, the first 170 entries of the input vector
# (context representation) are replaced with their mean value.
#
# The modified inputs are evaluated on the test dataset to compute
# top-k prediction accuracy.

accuracy_top_1  = 0
accuracy_top_10 = 0
window_size   = 1000
context_start = 170
eps = 1e-6

model.load_weights(f"{model_path}best_model.keras")
alpha         = np.load(f'{model_path}best_alpha.npy')
encoding      = np.load(f"{path_encoding}vocabulary_encoding.npy")
encoding_bits = ((encoding[:, None] >> np.arange(48)) & 1).astype(np.int8)

# compute mean context vector from test set for substitution ablation
mean_subs = np.load(f"{path_data}stochastic_input.npy").astype(np.float32)[5000:, :context_start].mean(axis=0)
N = 0

for i in range(5000, 10000, window_size):
    X = np.load(f"{path_data}stochastic_input.npy").astype(np.float32)
    Y = np.load(f"{path_data}stochastic_output.npy")
    X = X[i:i+window_size].copy()
    Y = Y[i:i+window_size].copy()

    # ablation: replace context vectors with their mean
    X[:, :context_start] = mean_subs

    preds = model.predict(X, verbose=0)
    preds = np.clip(preds, eps, 1 - eps)

    logit = np.log(preds) - np.log(1 - preds)  # (N_preds, bits)
    bias  = np.log(1 - preds)                   # (N_preds, bits)

    # apply bit weights
    logit *= alpha
    bias  *= alpha

    # compute scores and top-10 predictions
    scores = logit @ encoding_bits.T + bias.sum(axis=1, keepdims=True)
    result = np.argsort(-scores, axis=1)[:, :10]

    out = np.array([[encoding[result[i][j]] for j in range(result.shape[1])] for i in range(result.shape[0])])

    accuracy_top_1  += (out[:, 0] == Y).mean() * X.shape[0]
    accuracy_top_10 += (out == Y[:, None]).any(axis=1).mean() * X.shape[0]
    N += X.shape[0]

accuracy_top_1  /= N
accuracy_top_10 /= N
print(f"Accuracy of top-1 predictions:  {100*accuracy_top_1:02.2f}%")
print(f"Accuracy of top-10 predictions: {100*accuracy_top_10:02.2f}%")


Accuracy of top-1 predictions:  16.60%
Accuracy of top-10 predictions: 24.00%


In [7]:
# =========================
# BASELINE EVALUATION (NO ABLATION)
# =========================
# This cell evaluates the model on the test dataset without modification
# of the input context vector.
#
# This serves as the reference performance for comparison with ablation
# experiments.

accuracy_top_1  = 0
accuracy_top_10 = 0
window_size   = 1000
eps = 1e-6

model.load_weights(f"{model_path}best_model.keras")
alpha         = np.load(f'{model_path}best_alpha.npy')
encoding      = np.load(f"{path_encoding}vocabulary_encoding.npy")
encoding_bits = ((encoding[:, None] >> np.arange(48)) & 1).astype(np.int8)
N = 0

for i in range(5000, 10000, window_size):
    X = np.load(f"{path_data}stochastic_input.npy").astype(np.float32)
    Y = np.load(f"{path_data}stochastic_output.npy")
    X = X[i:i+window_size].copy()
    Y = Y[i:i+window_size].copy()

    preds = model.predict(X, verbose=0)
    preds = np.clip(preds, eps, 1 - eps)

    logit = np.log(preds) - np.log(1 - preds)  # (N_preds, bits)
    bias  = np.log(1 - preds)                   # (N_preds, bits)

    # apply bit weights
    logit *= alpha
    bias  *= alpha

    # compute scores and top-10 predictions
    scores = logit @ encoding_bits.T + bias.sum(axis=1, keepdims=True)
    result = np.argsort(-scores, axis=1)[:, :10]

    out = np.array([[encoding[result[i][j]] for j in range(result.shape[1])] for i in range(result.shape[0])])

    accuracy_top_1  += (out[:, 0] == Y).mean() * X.shape[0]
    accuracy_top_10 += (out == Y[:, None]).any(axis=1).mean() * X.shape[0]
    N += X.shape[0]

accuracy_top_1  /= N
accuracy_top_10 /= N
print(f"Accuracy of top-1 predictions:  {100*accuracy_top_1:02.2f}%")
print(f"Accuracy of top-10 predictions: {100*accuracy_top_10:02.2f}%")

Accuracy of top-1 predictions:  17.14%
Accuracy of top-10 predictions: 24.78%
